# HaGRID Gesture Classifier

Layer 3b of the vision pipeline. 18 classes, 64x64 hand crops, from-scratch CNN.

In [1]:
import os, json, math, random, hashlib
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score

In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")

device: mps


In [3]:
DATA_ROOT = "training data/hagrid-sample-30k-384p"
ANN_DIR   = f"{DATA_ROOT}/ann_train_val"
IMG_ROOT  = f"{DATA_ROOT}/hagrid_30k"
IMG_DIR_PREFIX = "train_val_"

CLASS_NAMES = [
    "call", "dislike", "fist", "four", "like", "mute", "ok", "one",
    "palm", "peace", "peace_inverted", "rock", "stop", "stop_inverted",
    "three", "three2", "two_up", "two_up_inverted",
]
NUM_CLASSES = 18
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IMG_SIZE = 64

CKPT_DIR = "checkpoints"
CKPT_PATH = f"{CKPT_DIR}/gesture_classifier_best.pt"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

assert len(CLASS_NAMES) == NUM_CLASSES
assert Path(ANN_DIR).is_dir(), f"missing {ANN_DIR} — is the working directory HAND_JOB?"
assert Path(IMG_ROOT).is_dir(), f"missing {IMG_ROOT}"
print("config ok")

config ok
